In [ ]:
chat_history = []

def add_user_message(message: str):

    chat_history.append(
        {
            "role": "user",
            "content": message
        }
    )
def add_assistant_message(message: str):

    chat_history.append(
        {
            "role": "assistant",
            "content": message
        }
    )
def get_chat_history():

    return chat_history

if __name__ == "__main__":

    add_user_message(
        "Hello"
    )

    add_assistant_message(
        "Hi"
    )

    print(
        get_chat_history()
    )

C:\Users\CA\AppData\Local\Temp\ipykernel_25276\1373621097.py:10: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(
C:\Users\CA\AppData\Local\Temp\ipykernel_25276\1373621097.py:16: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  self.collection=Collection("documents")
C:\Users\CA\AppData\Local\Temp\ipykernel_25276\1373621097.py:17: PyMilvusDeprecationWarning: `Collection.load` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  self.collection.load()


Retriever Initialized Successfully

Query:What is Milvus?

Embedding


In [2]:
from pymilvus import(
    connections,Collection
)
from app.ingestion.embedder import get_embedding

class Retriever:
    def __init__(self):
        #connect milvs

        connections.connect(
            alias="default",
            host="localhost",
            port="19530"
        )
        #load collection
        self.collection=Collection("documents")
        self.collection.load()
        print("Retriever Initialized Successfully")
    
    def embed_query(self,query:str):
        embedding=get_embedding(query)

        
        print("Query :", query)
        print("Type :", type(embedding))
        print("Length :", len(embedding))
        print("First Value Type :", type(embedding[0]))

        return embedding

    def search(self, query_embedding, k=5):

        results = self.collection.search(
            data=[query_embedding],
            anns_field="embedding",
            param={
                "metric_type": "COSINE"
            },
            limit=k,
            output_fields=["text"]
        )

        return results
if __name__ == "__main__":

    retriever = Retriever()
    query = "What is Remote Work Policy?"
    embedding = retriever.embed_query(query)
    results = retriever.search(
        embedding,
        k=3
    )

    for hit in results[0]:

        print("=" * 50)
        print("Score :", hit.score)
        print("Text :")
        print(hit.entity.get("text"))

C:\Users\CA\AppData\Local\Temp\ipykernel_7576\566310213.py:10: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(
C:\Users\CA\AppData\Local\Temp\ipykernel_7576\566310213.py:16: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  self.collection=Collection("documents")
C:\Users\CA\AppData\Local\Temp\ipykernel_7576\566310213.py:17: PyMilvusDeprecationWarning: `Collection.load` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  self.collection.load()


Retriever Initialized Successfully
Query : What is Remote Work Policy?
Type : <class 'list'>
Length : 1024
First Value Type : <class 'float'>
Score : 0.6896669268608093
Text :
arrangements may be approved by management. Employees must accurately record attendance
using approved systems.
6. Attendance and Punctuality
Employees are expected to arrive on time and maintain regular attendance. Repeated lateness,
absenteeism, or attendance violations may result in corrective action.
7. Remote Work Policy
Remote employees must maintain productivity, data security, and availability during working hours.
Company devices should be used for official business. Sensitive data must not be shared through
unauthorized channels.
8. Compensation and Payroll
Salaries are paid monthly. Payroll deductions may include taxes, benefits, and other authorized
deductions. Compensation reviews may occur annually based on performance and business
conditions.
Score : 0.5973711609840393
Text :
application, decision-

C:\Users\CA\AppData\Local\Temp\ipykernel_7576\566310213.py:33: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = self.collection.search(


In [1]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="phi3:mini"
)


def generate_answer(context: str, question: str):

    prompt = f"""
You are a helpful AI assistant.

Answer ONLY from the given context.

If the answer is not present in the context, say:
I could not find the answer in the document.

Context:
{context}

Question:
{question}

Answer:
"""

    response = llm.invoke(prompt)
    return response


# Temporary Testing
if __name__ == "__main__":

    context = """
    Remote employees must maintain productivity,
    data security, and availability during working hours.
    Company devices should be used for official business.
    """

    question = "What is Remote Work Policy?"

    answer = generate_answer(
        context,
        question
    )

    print("\nAnswer:\n")
    print(answer)

c:\Users\CA\Documents\rag-chatbot\backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Answer:

Remote Employees MUST adhere to the following rules at all times while on duty from home or another remote location, as per our company's policies which include maintaining productivity and data security. Company-issued devices should solely be utilized for official business purposes during working hours.


In [6]:
from app.retrieval.retriever import Retriever
from app.llm.generator import generate_answer


retriever = Retriever()


def ask_question(question: str):

    contexts = retriever.retrieve(
        question
    )

    context = "\n".join(contexts)

    answer = generate_answer(
        context=context,
        question=question
    )

    return answer
if __name__ == "__main__":

    question = "What is Remote Work Policy?"

    answer = ask_question(
        question
    )

    print("\nAnswer:\n")
    print(answer)

Retriever Initialized Successfully


c:\Users\CA\Documents\rag-chatbot\backend\app\retrieval\retriever.py:10: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(
c:\Users\CA\Documents\rag-chatbot\backend\app\retrieval\retriever.py:16: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  self.collection=Collection("documents")
c:\Users\CA\Documents\rag-chatbot\backend\app\retrieval\retriever.py:17: PyMilvusDeprecationWarning: `Collection.load` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  self.collection.load()


AttributeError: 'Retriever' object has no attribute 'retrieve'